In [2]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/ML-FinalAss/Walmart-Sales-Forecasting"

!pip install -U kaggle mlflow dagshub

Mounted at /content/drive
/content/drive/MyDrive/ML-FinalAss/Walmart-Sales-Forecasting
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.5/111.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:

import os
import zipfile
import glob

data_dir = "data"
required_files = ["train.csv", "test.csv", "stores.csv", "features.csv"]

files_exist = os.path.exists(data_dir) and all(
    os.path.exists(os.path.join(data_dir, f)) for f in required_files
)

if files_exist:
    print("Dataset is already downloaded and extracted in the 'data' folder. Skipping download!")
else:
    print("Dataset not found. Starting download...")

    os.environ['KAGGLE_API_TOKEN'] = "KGAT_caf00baa57d3f767dc15d31b24d4e730"

    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting

    os.makedirs(data_dir, exist_ok=True)
    main_zip = "walmart-recruiting-store-sales-forecasting.zip"
    if os.path.exists(main_zip):
        with zipfile.ZipFile(main_zip, "r") as zip_ref:
            zip_ref.extractall(data_dir)
        os.remove(main_zip)
    else:
        print("The zip file wasn't found. Double check your Kaggle Token expiration status!")

    inner_zips = glob.glob(os.path.join(data_dir, "*.zip"))
    for file in inner_zips:
        with zipfile.ZipFile(file, "r") as zip_ref:
            zip_ref.extractall(data_dir)
        os.remove(file)

    print("Success! All datasets downloaded and extracted:")
    print(os.listdir(data_dir))

Dataset is already downloaded and extracted in the 'data' folder. Skipping download!


In [4]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import dagshub

dagshub.init(repo_owner='slosa23', repo_name='Walmart-Sales-Forecasting', mlflow=True)
mlflow.set_experiment("Ensemble_Blending_V1")

train_raw = pd.read_csv("data/train.csv")
stores = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=7a31f84a-6196-4f07-93d8-ed7cd2e738d3&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=3babbe7481f648af9d8b4061bddb4f4de8f6be5cb8d7aeb544a075583990b14e




Accessing as slosa23

Initialized MLflow to track repo "slosa23/Walmart-Sales-Forecasting"

Repository slosa23/Walmart-Sales-Forecasting initialized!

In [7]:
import yaml
import mlflow.pytorch
import mlflow.sklearn
import torch
import numpy as np
from sklearn.preprocessing import RobustScaler
from src.features.preprocessing import get_model_ready_data
from src.evaluation.metrics import calculate_wmae
from src.models.ensemble_utils import calculate_blended_predictions

with open("configs/ensemble_config.yaml", "r") as f:
    config = yaml.safe_load(f)

split_date = config['data']['split_date']
v1_seq_len = config['model']['params']['seq_len']
v1_pred_len = config['model']['params']['pred_len']
weight_options = config['ensemble']['search_weights']

with mlflow.start_run(run_name="Optimized_V2_Ensemble_Blend") as run:
    print(f"Started tracking run: {run.info.run_id}")

    print("Processing structural features and validation partitions...")
    X_train, y_train, X_val, y_val, is_holiday_val, preprocessor = get_model_ready_data(
        train_raw, stores, features, split_date=split_date
    )

    XGB_URI = "models:/Walmart_XGBoost_Baseline/1"
    DLINEAR_V1_URI = "models:/Walmart_DLinear_Baseline/1"

    print(f"Downloading registered XGBoost from: {XGB_URI}")
    xgb_pipeline = mlflow.sklearn.load_model(XGB_URI)
    xgb_preds = xgb_pipeline.predict(X_val)

    print(f"Downloading registered DLinear (Version 1) from: {DLINEAR_V1_URI}")
    dl_model = mlflow.pytorch.load_model(DLINEAR_V1_URI)
    dl_model.eval()

    target_scaler = RobustScaler()
    y_train_scaled = target_scaler.fit_transform(y_train.values.reshape(-1, 1)).flatten()

    with torch.no_grad():
        lookback_window = torch.tensor(y_train_scaled[-v1_seq_len:], dtype=torch.float32).view(1, -1, 1)
        raw_dl_forecast = dl_model(lookback_window).numpy().flatten()

    dl_scaled_preds = np.tile(raw_dl_forecast, int(np.ceil(len(y_val) / v1_pred_len)))[:len(y_val)]
    dl_preds = target_scaler.inverse_transform(dl_scaled_preds.reshape(-1, 1)).flatten()

    print("\n uning Blending Ratios across uniform matrix footprints:")
    best_wmae = float('inf')
    best_weights = None
    best_blend_preds = None

    for w in weight_options:
        current_weights = {'xgb_weight': w, 'lgb_weight': 0.0, 'nn_weight': round(1.0 - w, 2)}
        blended_output = calculate_blended_predictions(xgb_preds, np.zeros_like(xgb_preds), dl_preds, current_weights)

        score = calculate_wmae(y_val, blended_output, is_holiday_val)
        print(f"   Ratio [{int(w*100)}% XGB / {int((1-w)*100)}% DLinear] -> Combined WMAE: {score:.2f}")

        if score < best_wmae:
            best_wmae = score
            best_weights = current_weights
            best_blend_preds = blended_output

    print(f"\n WINNING COMBINATION METRICS:")
    print(f"Absolute Lowest Validation WMAE: {best_wmae:.2f}")

    mlflow.log_params(best_weights)
    mlflow.log_metric("WMAE", best_wmae)

    mlflow.sklearn.log_model(
        sk_model=xgb_pipeline,
        artifact_path="final_ensemble_model",
        registered_model_name="Walmart_Final_Ensemble_Blend",
        serialization_format="pickle"
    )

    np.save("final_ensemble_predictions.npy", best_blend_preds)
    mlflow.log_artifact("final_ensemble_predictions.npy")

    print("Entire ensemble process tracked, saved, and registered on DagsHub")

Started tracking run: 81def0e0f4884756adcb1007e331d0a8
Processing structural features and validation partitions...



 uning Blending Ratios across uniform matrix footprints:
   Ratio [30% XGB / 70% DLinear] -> Combined WMAE: 13012.61
   Ratio [40% XGB / 60% DLinear] -> Combined WMAE: 12951.64
   Ratio [50% XGB / 50% DLinear] -> Combined WMAE: 12977.17
   Ratio [60% XGB / 40% DLinear] -> Combined WMAE: 13079.31
   Ratio [70% XGB / 30% DLinear] -> Combined WMAE: 13251.17

 WINNING COMBINATION METRICS:
Absolute Lowest Validation WMAE: 12951.64


2026/07/10 13:56:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/10 13:56:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'Walmart_Final_Ensemble_Blend'.
2026/07/10 13:56:46 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Walmart_Final_Ensemble_Blend, version 1
Created version '1' of model 'Walmart_Final_Ensemble_Blend'.


Entire ensemble process tracked, saved, and registered on DagsHub
🏃 View run Optimized_V2_Ensemble_Blend at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/3/runs/81def0e0f4884756adcb1007e331d0a8
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/3
